# R4 · 2 · MIX_elastico e DoisEstagios_det

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem tudo das rodadas 1 a 3):
1. suba **`r4_lib.py`** (pasta `data_imdc_2026/RODADA4/`) e confira que `r3_lib.py` e `r2_lib.py` estão lá;
2. as pastas `resultados_uf_r4`, `revisoes_r4`, `cache_r4` e `avaliacao_r4` são criadas sozinhas.

**Usa a rodada 3:** o normal (ENS_grafos3_λ) vem de `cache_r3/normal_ens_grafos3_bb_clim.parquet` e as misturas da R3 entram na comparação.

**Complementar:** nada das rodadas anteriores muda. As previsões oficiais (corte de junho) vão para `resultados_uf_r4/uf/`; as revisões de julho a outubro, para `revisoes_r4/uf/`; as tabelas, para `avaliacao_r4/` (com data e hora no nome e uma cópia `_ultimo`).

**Retomada:** se a internet ou a luz cair, reconecte e **rode tudo de novo**. Cada etapa grava o seu arquivo (mar, sorotipo por ano, variáveis, detector por método × corte, parâmetros, um arquivo por modelo × temporada, os pesos das redes a cada 5 épocas, um PNG por UF) e o que já existe é pulado.

CPU comum, 20–40 min. **Rode depois do R4_1.** Pode rodar **ao mesmo tempo** que o R4_3 (GPU).

- **`MIX_elastico_UF`:** corrige o defeito do `MIX_escala_UF` da R3. Ele acertava a altura do pico, mas inflava a entressafra (+156%) porque multiplicava a curva inteira pelo mesmo fator.
  - Agora cada semana × UF cresce conforme a sua **elasticidade histórica**, que depende do **nível normal** daquela semana: quanto o log da incidência sobe quando a temporada é maior, medido nas temporadas já terminadas.
  - No histórico até 2021, ela vale ~0,1 nas semanas quase sem casos e ~1 no pico. O total continua o do detector.
  - O *w* (0,5, 0,75 ou 1: quanto confiar no detector) é escolhido na afinação pelo **critério misto** = ½ R × climatologia + ½ WIS absoluto relativo.
- **`DoisEstagios_det_UF`:** o *retreino do item 1*.
  - Tamanho: o do detector ampliado.
  - Forma: sorteada entre temporadas passadas **parecidas em tamanho**, da própria UF (peso 1) ou da macrorregião (peso 0,4). Assim, quando o detector diz "ano grande", entram as formas dos anos grandes (início mais cedo, pico mais estreito).
  - Ainda: deslocamento de ±2 semanas e ruído binomial negativo.
- **Revisões:** o `MIX_elastico_UF` é refeito com o detector de jul/ago/set/out e vai para `revisoes_r4/`.


In [ ]:
NOTEBOOK = 'r4_mistura'

# @title 1 · Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/r4', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'r2_lib.py', 'r3_lib.py',
                 'r4_lib.py', 'dados', 'dados_uf', 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    pass
sys.path.insert(0, PASTA_LOCAL)
PASTA_R4 = os.path.join(PASTA_DRIVE, 'resultados_uf_r4')
PASTA_REV4 = os.path.join(PASTA_DRIVE, 'revisoes_r4')
PASTA_C4 = os.path.join(PASTA_DRIVE, 'cache_r4')
PASTA_AV4 = os.path.join(PASTA_DRIVE, 'avaliacao_r4')
PASTA_R3 = os.path.join(PASTA_DRIVE, 'resultados_uf_r3')
PASTA_C3 = os.path.join(PASTA_DRIVE, 'cache_r3')
PASTA_AV2 = os.path.join(PASTA_DRIVE, 'avaliacao_r2')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
PASTA_EXT = os.path.join(PASTA_DRIVE, 'resultados_externos')
for p in (os.path.join(PASTA_R4, 'uf'), os.path.join(PASTA_REV4, 'uf'), PASTA_C4, os.path.join(PASTA_C4, 'ckpt'), PASTA_AV4):
    os.makedirs(p, exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U, semanal_lib as S, r2_lib as R2, r3_lib as R3, r4_lib as R4
for m in (V, L, CM, U, S, R2, R3, R4):
    importlib.reload(m)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
D = ctx.Du
TUNING, TESTE = U.TUNING, U.TESTE
REFAZER = False
R4.PESO_ABS = 0.5     # critério misto = (1 − p)·R × referência + p·WIS absoluto relativo. Suba (ex.: 0.7) para priorizar a altura dos picos


In [ ]:
# @title 2 · Normal (cache da R3) e detector ampliado
NORMAL = R4.ler_normal(ctx, PASTA_DRIVE, PASTA_AV2, PASTA_COMP, PASTA_EXT, PASTA_C3, TUNING + TESTE)
ESC, DET = R4.detector_escolhido4(PASTA_C4)
print(NORMAL.groupby('modelo').season.unique(), ESC)


In [ ]:
# @title 3 · w do MIX_elastico (afinação, critério misto; gravado em json)
W = R4.escolher_w_elastico(ctx, NORMAL, DET, PASTA_C4, M=500)
print('w escolhido:', W['w'])
pd.DataFrame(W['tabela'])


In [ ]:
# @title 4 · Gerar MIX_elastico_UF (jun + revisões) e DoisEstagios_det_UF (um arquivo por modelo × temporada)
TAREFAS = [('MIX_elastico_UF', c) for c in R4.CORTES] + [('DoisEstagios_det_UF', 'jun')]
for nome0, corte in TAREFAS:
    nome = nome0 if corte == 'jun' else f'{nome0}@{corte}'
    pasta = PASTA_R4 if corte == 'jun' else PASTA_REV4
    falta = [s for s in TUNING + TESTE if REFAZER or not os.path.exists(L._arq(pasta, 'uf', nome, s))]
    if not falta:
        continue
    t0 = time.time()
    if nome0 == 'MIX_elastico_UF':
        x = R4.gerar_mix_elastico(ctx, NORMAL, DET, W['w'], corte, falta, M=1000)
    else:
        x = R4.gerar_dois_estagios_det(ctx, DET, corte, falta, M=1000)
    if x is None:
        print('sem dados:', nome); continue
    R4.grava_modelo(x, pasta, nome)
    print(f'✅ {nome:28s} {sorted(x.season.unique())} · {time.time() - t0:.0f}s', flush=True)


In [ ]:
# @title 5 · Conferência rápida no teste (o veredito completo é no R4_4)
X = pd.concat([NORMAL, R4.ler_pastas(PASTA_R4)], ignore_index=True)
X = X[X.season.isin(TESTE) & X.modelo.isin(['ENS_grafos3_λ', 'BB', 'MIX_elastico_UF', 'DoisEstagios_det_UF'])]
R4.criterio(ctx, X, ref='BB').round(3)
